In [1]:
%load_ext autoreload
%autoreload 2

# Product Confirmation Workflow - Direct S3 Access

This notebook reads DIST-ALERT products directly from S3 and runs the confirmation workflow without downloading.

In [2]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow
from utils_s3 import run_confirmation_wrapper_parallel

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
csv_path = Path('final_run/val_products_transformer_optimized-max10_processed_2026-02-05.csv')
df = pd.read_csv(csv_path)
df.head()

,name,zip_url,prod_dir,s3_prod_uri,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,...,memory_strategy,batch_size_for_norm_param_estimation,model_context_length,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,builtnewalert__60HTF__transformer_optimized-10...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2026-02-05T19:02:43+00:00,846.634,4.5,60HTF,1,...,high,32,10,2024-12-09,146,2.5,4,best,none,False
1,other__54HXH__transformer_optimized-10mc_val_s...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2026-02-05T19:02:43+00:00,596.966,4.5,54HXH,1,...,high,32,10,2024-05-10,16,2.5,4,best,none,False
2,other__54HXH__transformer_optimized-10mc_val_s...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2026-02-05T19:02:43+00:00,924.449,4.5,54HXH,1,...,high,32,10,2024-10-18,89,2.5,4,best,none,False
3,builtnewalert__60HTF__transformer_optimized-10...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2026-02-05T19:02:43+00:00,400.146,4.5,60HTF,1,...,high,32,10,2024-02-02,154,2.5,4,best,none,False
4,other__54HXH__transformer_optimized-10mc_val_s...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2026-02-05T19:02:43+00:00,883.056,4.5,54HXH,1,...,high,32,10,2024-07-14,89,2.5,4,best,none,False


## Group products by event and MGRS tile

In [4]:
# Extract event name from job_name
df['identifier'] = df['name'].map(lambda x: '__'.join(x.split('__')[:2]))

# Group by event and MGRS tile
grouped = df.groupby(['identifier', 'mgrs_tile_id'])

print(f"Number of event-tile combinations: {len(grouped)}")
print("\nEvent-tile combinations:")
for (event, tile), df_group in grouped:
    print(f"  {event} / {tile}: {len(df_group)} products")

Number of event-tile combinations: 85

Event-tile combinations:
  builtnewalert__14RNU / 14RNU: 58 products
  builtnewalert__16SBF / 16SBF: 54 products
  builtnewalert__22KFA / 22KFA: 30 products
  builtnewalert__22LDQ / 22LDQ: 39 products
  builtnewalert__22MDT / 22MDT: 47 products
  builtnewalert__22MGB / 22MGB: 61 products
  builtnewalert__43QEU / 43QEU: 48 products
  builtnewalert__43QFE / 43QFE: 48 products
  builtnewalert__44PKB / 44PKB: 56 products
  builtnewalert__50SMF / 50SMF: 83 products
  builtnewalert__52TDP / 52TDP: 51 products
  builtnewalert__60HTF / 60HTF: 119 products
  cropnew__14SPH / 14SPH: 52 products
  cropnew__17TPG / 17TPG: 54 products
  cropnew__20HMD / 20HMD: 61 products
  cropnew__20HMJ / 20HMJ: 32 products
  cropnew__21LWC / 21LWC: 29 products
  cropnew__31PGN / 31PGN: 55 products
  cropnew__35MRM / 35MRM: 58 products
  cropnew__36RTS / 36RTS: 92 products
  cropnew__41UPB / 41UPB: 2 products
  fire__17UPT / 17UPT: 30 products
  fire__20LNJ / 20LNJ: 28 produ

## Run Confirmation Workflow

For each event-tile combination, run the sequential confirmation workflow using S3 URIs directly.

In [5]:
df_group.columns

Index(['name', 'zip_url', 'prod_dir', 's3_prod_uri', 'browse_url',
       'product_request_time', 'processing_duration',
       'high_confidence_alert_threshold', 'mgrs_tile_id',
       'post_date_buffer_days', 'stride_for_norm_param_estimation',
       'n_workers_for_norm_param_estimation', 'delta_lookback_days_mw',
       'use_date_encoding', 'model_source', 'memory_strategy',
       'batch_size_for_norm_param_estimation', 'model_context_length',
       'post_date', 'track_number', 'low_confidence_alert_threshold',
       'n_workers_for_despeckling', 'device', 'max_pre_imgs_per_burst_mw',
       'model_compilation', 'identifier'],
      dtype='object')

In [8]:
confirmation_input_data = [(f'out/{csv_path.stem}/{identifier}', df_group.s3_prod_uri.tolist()) for (identifier, _), df_group in grouped]
confirmation_input_data[0]

('out/val_products_transformer_optimized-max10_processed_2026-02-05/builtnewalert__14RNU',
 ['s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/fb0edac8-9a30-4e4c-af0c-46a0b3014da0/OPERA_L3_DIST-ALERT-S1_T14RNU_20240111T004342Z_20260205T204727Z_S1A_30_v0.1',
  's3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/3e88b762-bc4d-40f6-9b83-5ad491eb9ba1/OPERA_L3_DIST-ALERT-S1_T14RNU_20240428T004342Z_20260205T204644Z_S1A_30_v0.1',
  's3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/49bf92ec-1b68-45f0-a8b9-c17d05f5a98f/OPERA_L3_DIST-ALERT-S1_T14RNU_20240517T003532Z_20260205T205218Z_S1A_30_v0.1',
  's3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/81920d2d-4598-47ca-b3ab-0141ee6053df/OPERA_L3_DIST-ALERT-S1_T14RNU_20240411T003530Z_20260205T205127Z_S1A_30_v0.1',
  's3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/86bcb6a6-4b5b-4536-b6a3-4eac7807fac3/OPERA_L3_DIST-ALERT-S1_T14RNU_20240510T004343Z_20260205T204543Z_S1A_30_v0.1',
  's3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/036f

In [9]:
run_confirmation_wrapper_parallel(confirmation_input_data)

Process SpawnPoolWorker-8:
  0%|          | 0/85 [07:11<?, ?it/s]


KeyboardInterrupt: 